## **<center>Exploring Electric Vehicle Registrations:</center>**
## **<center>Trends in Adoption, Manufacturers, and Electric Range</center>**



## Executive Summary


This project explores the distribution and characteristics of electric vehicles (EVs) using a large dataset of registered vehicles. The analysis focuses on understanding trends in vehicle types, manufacturers, geographic distribution, and electric range. The results show that Battery Electric Vehicles (BEVs) dominate the market, with Tesla leading registrations by a significant margin. EV adoption has increased rapidly in recent years and is concentrated in a few key counties. While the dataset provides strong insights into EV patterns, certain limitations—such as unclear CAFV eligibility categories—affect deeper analytical interpretation.

## Table of Contents

1. Introduction

2. Data Loading

3. Dataset Overview

4. Data Cleaning

    4.1. Schema and Structural Normalization

5. Exploratory Data Analysis (Univariate)

6. Bivariate Analysis

7. Feature Relationships

8. Conclusion

9. References


## 1. Introduction

The transition to electric vehicles is a key component of global efforts to reduce emissions and promote sustainable transportation. This exploratory data analysis (EDA) examines a dataset of electric vehicle registrations to better understand market trends and adoption patterns. The analysis focuses on key features such as vehicle type (BEV vs PHEV), manufacturer, model, electric range, and geographic distribution.

The objective of this project is not to build predictive models, but to explore the data, identify patterns, and generate meaningful insights about how electric vehicles are distributed across regions and manufacturers.

## 2. Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

url = "https://data.wa.gov/api/views/f6w7-q2d2/rows.csv?accessType=DOWNLOAD"

battery = pd.read_csv(url)




## 3. Dataset Overview

In [ ]:
print('=== SHAPE OF DATASET ===')
row, col = battery.shape; print(f"Rows: {row}"), print(f"Columns: {col}")
print(f"Memory Usage:{battery.memory_usage(deep=True).sum() /1024**2:.2f} MB")

print('_'*120)
print("Data Validation Summary".center(120))
print("_"*120)

summary = pd.DataFrame({
    'DataType': battery.dtypes,
    'Non-Null': battery.count(),
    'Null': battery.isnull().sum(),
    'Null%': (battery.isnull().sum() / len(battery) * 100).round(2),
    'Unique': battery.nunique(),
    'Duplicates': battery.duplicated().sum()
})

pd.set_option("display.max_rows", 100)
summary

**Summary of your DataFrame**

In [ ]:
battery.info()

In [ ]:
battery[['Model Year','Electric Range']].describe()

* These summary statistics show that most vehicles in the dataset are quite new, with model years clustered around 2022–2024, which suggests recent growth in EV adoption. Electric range varies widely: while some vehicles can go up to 337 miles, both the 25th percentile and median are 0, indicating that a large portion likely consists of plug-in hybrids or vehicles with little to no electric-only range.


## 4. Data Cleaning

### 4.1. Schema and Structural Normalization

In [ ]:
battery.columns

**Convert Column Names to snake_case**

In [ ]:
def convert_col(df):
    df.columns = (df.columns
            .str.strip()
            .str.replace(" ", "_")
            .str.lower())
    return df
battery = convert_col(battery)

In [ ]:
for col in battery.columns:
    print(col)

**Dropping Unnecessary Columns**

In [ ]:
def dropping_col(df):
    return df.drop(columns=['postal_code', 
                            '2020_census_tract', 
                            'vehicle_location', 
                            'dol_vehicle_id' ])
    
battery = dropping_col(battery)

* The ***2020_census_tract*** and ***postal_code*** were removed as they add unnecessary geographic detail for this analysis. Dropping these columns helps reduce noise and keeps the focus on features that meaningfully provide analytical values. Moreover, it saves memory and speeds up processing.

* The ***vehicle_location*** column was removed because this analysis does not include spatial or map-based exploration.
* The ***DOL Vehicle ID*** column was removed because it serves only as a unique administrative identifier and does not contribute to analytical insights or modeling objectives.

**Handling Null Values**

In [ ]:
(
    battery[['county',
          'city', 
          'legislative_district',
          'electric_range',
          'electric_utility']]
    .isna()
    .sum()
    * 100 / len(battery)
).round(4)

* The proportion of missing values in each column is well below 0.5%, making their impact on the overall dataset negligible. Removing these observations would lead to unnecessary data loss without a meaningful improvement in data quality. Therefore, the missing values are retained and will be handled as appropriate within the analysis context.

**Cleaning Non-U.S. State Entries in the State Column**

In [ ]:
for k,v in battery.state.value_counts().items():
    print(k,v, end="| ")

In [ ]:
(
    battery.drop(battery.loc[battery
                 .state.isin(["BC", "QC", "AB", "AE", "AP", "Gu"])]
                 .index, 
                 inplace=True))
battery.loc[battery.state.isin(["BC", "QC", "AB", "AE", "AP", "Gu"])].shape

* The dataset contains several entries that are not U.S. states, including Canadian provinces (BC, QC, AB), military address codes (AE, AP), and the U.S. territory of Guam (GU).
* These entries were excluded to maintain geographic consistency and improve the accuracy and interpretability of the analysis.


## 5. Exploratory Data Analysis (Univariate)

In [ ]:
plt.figure(figsize=(6, 3), dpi=120)

ax= sns.histplot(battery['electric_range'], 
             bins=30,
             color='skyblue', 
             line_kws={'linewidth': 2, 'color': 'darkred'},
             kde=True,
             linewidth=0.3, 
             fill=True)

ax.tick_params(axis='y', labelsize=8)
ax.tick_params(axis='x', labelsize=8)

plt.title('Distribution of Electric Range', 
          size=11,
          fontfamily='sans-serif')
plt.xlabel('Electric Range', 
           size=8, 
           fontfamily='sans-serif')
plt.ylabel('Count', 
           size=8, 
           fontfamily='sans-serif')
plt.tight_layout()
plt.show()

* The distribution of electric range is highly skewed, with a strong concentration near 0 miles. This indicates that a significant portion of the dataset consists of plug-in hybrid vehicles or models with minimal electric-only capability. In contrast, a secondary cluster appears between approximately 200–250 miles, representing newer fully electric vehicles with substantially higher range.
* The clear separation between these groups suggests a transitional market structure, where older or hybrid vehicles coexist with modern long-range EVs, reflecting the ongoing shift toward full electrification.

In [ ]:
plt.rcParams['figure.dpi'] = 120
top_cities = battery['city'].value_counts(ascending=True).tail(10)

ax = top_cities.plot(kind='barh', figsize=(7,4))

total = sum([bar.get_width() for bar in ax.patches])  

for bar in ax.patches:
    width = bar.get_width()
    y = bar.get_y() + bar.get_height() / 2
    percent = (width / total) * 100
    
    ax.text(width, y, f'{percent:.1f}%', va='center', color='black', fontsize=9)


plt.title('Top Cities by EV Registrations (Share %)', 
          fontsize=11,
          fontfamily='sans-serif')
plt.xlabel('Number of Vehicles', 
           fontsize=9,
           fontfamily='sans-serif')
plt.ylabel('Cities', 
           fontsize=9,
           fontfamily='sans-serif')

ax.tick_params(axis='y', labelsize=8)
ax.tick_params(axis='x', labelsize=8)
plt.tight_layout()

* After analyzing electric range, we now examine top cities. Seattle, Bellevue, and Vancouver account for the highest concentrations of registered electric vehicles, with Seattle alone representing 35.4% of total registrations. This strong urban clustering suggests that EV adoption is closely tied to infrastructure availability, policy incentives, and population density.
* The dominance of these cities highlights how supportive ecosystems—such as charging networks and local regulations—play a critical role in accelerating EV adoption.

In [ ]:
top10 = battery['county'].value_counts().head(10)
plt.figure(figsize=(6, 4), dpi=120)

ax = sns.countplot(x=battery['county'], order=top10.index, color='#00d4aa')

total = top10.sum()  

for bar in ax.patches:
    height = bar.get_height()
    x = bar.get_x() + bar.get_width() / 2
    
    percent = (height / total) * 100
    
    ax.text(x, height, f'{percent:.1f}%', 
            ha='center', 
            va='bottom', 
            color='black', 
            fontsize=9)

ax.tick_params(axis='y', labelsize=8)
ax.tick_params(axis='x', labelsize=8)

plt.title('Leading Counties by Number of Registered Electric Vehicles', 
          fontsize=11, 
          fontfamily='sans-serif')
plt.xlabel('County', 
           fontsize=9,
           fontfamily='sans-serif')
plt.ylabel('Number of Vehicles', 
           fontsize=9,
           fontfamily='sans-serif')
plt.xticks(rotation=30)

plt.tight_layout()
plt.show()

* Next, we look at top counties. King, Snohomish, and Pierce counties have the highest number of registered electric vehicles in this dataset. This indicates a higher concentration of EV registrations in these counties compared to others. Therefore, it suggests greater adoption of electric vehicles at the county level.
* The percentages shown are calculated relative to the top 10 counties, rather than the entire dataset. This approach provides a clearer comparison of how these leading counties contribute within the selected group. This approach is not influenced by smaller, less significant categories outside the top 10.

In [ ]:
top5= battery.make.value_counts().head(10)

plt.figure(figsize=(6, 4), dpi=120)

ax = sns.countplot(x=battery['make'], order=top5.index, color='#3b82f6')

ax.tick_params(axis='y', labelsize=8)
ax.tick_params(axis='x', labelsize=8)

plt.title('Top manufacturers by number of registered electric vehicles',
          fontsize=11,
          fontfamily='sans-serif')

plt.xlabel('Manufacturer',
           fontsize=9,
           fontfamily='sans-serif')
plt.ylabel('Number of Registered Vehicles',
           fontsize=9,
           fontfamily='sans-serif')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

* After analyzing top counties, we now examine the top manufacturers dominating the market. Tesla, Chevrolet, and Nissan rank among the top manufacturers by EV registrations; however, Tesla’s lead is substantial, exceeding 100,000 registrations while its closest competitors remain below 20,000.

* This disparity underscores Tesla’s dominant market position, reflecting the advantages of early adoption, strong brand equity, and an integrated charging ecosystem, which together have enabled it to outpace traditional automakers in the transition to electric mobility.

In [ ]:
plt.rcParams['figure.dpi'] = 120
counts = battery.model.value_counts().sort_values(ascending=True).tail(10)

ax = counts.plot(kind='barh', 
                 figsize=(7,4),
                 color="#4ee184",
                 width=0.7)

total = sum([bar.get_width() for bar in ax.patches])  

for bar in ax.patches:
    width = bar.get_width()
    y = bar.get_y() + bar.get_height() / 2
    percent = (width / total) * 100
    
    ax.text(width, y, f'{percent:.1f}%', va='center', color='black', fontsize=9)

ax.tick_params(axis='y', labelsize=8)
ax.tick_params(axis='x', labelsize=8)

plt.title('Top Models by number of registered electric vehicles', 
          fontsize=11,
          fontfamily='sans-serif')
plt.xlabel('Model',
           fontsize=9,
           fontfamily='sans-serif')
plt.ylabel('Number of Registered Vehicles',
           fontsize=9,
           fontfamily='sans-serif')

plt.xticks(rotation=0)
plt.tight_layout()


* With regard to the electric vehicle models, the Tesla Model Y and Model 3, alongside the Nissan Leaf, account for the highest number of registrations, with the Model Y leading by nearly 60,000 units.
* This pattern reflects strong consumer demand for Tesla’s scalable, mass-market offerings, while the sustained presence of the Nissan Leaf underscores its role as a foundational model in early EV adoption. Together, these trends highlight the importance of affordability, reliability, and brand trust in driving large-scale EV adoption.

In [ ]:
evt = battery['electric_vehicle_type'].value_counts()
label = ['(BEV)', '(PHEV)' ]
plt.rcParams['figure.dpi']=270
plt.figure(figsize=(5,5))

plt.pie(evt, 
        labels=label, 
        textprops={'fontsize': 8},
         autopct='%.0f%%',
         colors=['#22c55e',"#5CF4D5"])

plt.title('BEV vs PHEV Registration Distribution',
          fontsize=11, 
          fontweight='bold',
          fontfamily='sans-serif')

plt.tight_layout()
plt.show()

* Battery Electric Vehicles (BEVs) account for nearly 80% of total registrations, significantly outpacing Plug-in Hybrid Electric Vehicles (PHEVs), which comprise the remaining 20%.
* This imbalance reflects a clear transition toward full electrification, driven by advancements in battery performance, expanded charging infrastructure, and increasing consumer willingness to adopt fully electric mobility over transitional hybrid solutions.

In [ ]:
plt.figure(figsize=(6, 4), dpi=120)
counts = battery.model_year.value_counts().sort_values(ascending=True)
ax = counts.plot(kind='bar',
                 color="#22d3ee",
                 width=0.8)

ax.plot(range(len(counts)), counts.values, color='#fde047', marker='o')


ax.tick_params(axis='y', labelsize=8)
ax.tick_params(axis='x', labelsize=8)

plt.title('Growth of Electric Vehicle in the last two decades',
          fontsize=11,
          fontfamily='sans-serif')
plt.xlabel('Model Year',
           fontsize=9,
           fontfamily='sans-serif')

plt.ylabel('Count',
           fontsize=9,
           fontfamily='sans-serif')

plt.xticks(rotation=35)

plt.tight_layout()
plt.show()

* The graph illustrates the growth of electric vehicle production over the past two decades. Before 2010, the number of electric vehicles produced was very low, with only a handful recorded each year. Starting around 2010, production began to increase steadily, rising from approximately 100 vehicles in 2011 to nearly 60,000 vehicles by 2023. This sharp upward trend highlights the rapid expansion of the electric vehicle market in recent years. It is likely driven by advancements in technology, environmental awareness, and stronger policy support for sustainable transportation.

In [ ]:
pd.DataFrame(
    battery.state
    .value_counts()
    .head(10)
)

* EV registrations are heavily concentrated in Washington State, with very limited representation from other states, indicating a strong regional bias in the dataset.

## 6. Bivariate Analysis

**Electric Range vs. Vehicle Type**

In [ ]:
plt.figure(figsize=(6,4), dpi=120)
ax = sns.violinplot(x='electric_vehicle_type', 
                    y='electric_range',
                    data=battery, 
                    color='#00d4aa')

ax.tick_params(axis='y', labelsize=8)
ax.tick_params(axis='x', labelsize=8)

plt.title('Electric Range Distribution by Vehicle Type', 
          fontsize=11)
plt.xlabel('Vehicle Type', 
           fontsize=9,
           fontfamily='sans-serif')

plt.ylabel('Electric Range (miles)', 
           fontsize=9,
           fontfamily='sans-serif')

plt.tight_layout()
plt.show()

* Battery Electric Vehicles (BEVs) exhibit a highly skewed distribution of electric range. Most vehicles are concentrated at lower ranges and a long tail extending toward higher values. This reflects the evolution of battery technology, where newer models offer significantly greater range.

* In contrast, Plug-in Hybrid Electric Vehicles (PHEVs) show a much narrower distribution, typically limited to shorter electric ranges. That's because it relies on hybrid power systems.

**Electric Range vs. Model Year**

In [ ]:
plt.figure(figsize=(6, 4), dpi=120)

ax = sns.violinplot(data=battery,
                  x='model_year',
                  y='electric_range',
                  color="#00d4ab",
                  )

ax.tick_params(axis='y', labelsize=8)
ax.tick_params(axis='x', labelsize=8)

plt.title('Distribution of Electric Range Across Years', 
          fontsize=11,
          fontfamily='sans-serif')
plt.ylabel('Electric Range',
           fontsize=9,
            fontfamily='sans-serif')
plt.xlabel('Model Year', 
           fontsize=9,
           fontfamily='sans-serif')

plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


* The violin plot illustrates the distribution of electric range across model years. Earlier years exhibit narrower distributions with generally lower values, indicating limited variation and lower battery capacity among early EV models. From around 2015 to 2020, both the median range and variability increase, reflecting the introduction of a wider mix of vehicles with varying battery capacities. However, the most recent years show compressed distributions with very low values. This more likely indicating missing or incomplete data rather than a true decline in electric range. Overall, the observed patterns suggest that variations in range are influenced by both technological advancements and the evolving mix of vehicle types, rather than a consistent upward trend.

**Electric Range vs. EV Type**

In [ ]:
plt.figure(figsize=(6, 4), dpi=120)

  

ax = sns.barplot(x=battery.electric_vehicle_type, 
            y=battery.electric_range,
            color='#3b82f6',
            width=0.5)

ax.tick_params(axis='y', labelsize=8)
ax.tick_params(axis='x', labelsize=8)

plt.title('Distribution of Electric Range Across the EV Type',
          fontsize=11, 
          fontfamily='sans-serif')

plt.xlabel('Type of Electric Vehicle', 
           size=8, 
           fontfamily='sans-serif')
plt.ylabel('Electric Range', 
           size=8, 
           fontfamily='sans-serif')

plt.tight_layout()
plt.show()

* Battery Electric Vehicles (BEVs) have a higher average electric range of approximately 42 miles, compared to 31–32 miles for Plug-in Hybrid Electric Vehicles (PHEVs).

* This gap highlights the structural differences between the two technologies, with BEVs engineered for full electric performance, while PHEVs balance electric and combustion systems. As a result, BEVs offer greater electric-only usability, reinforcing their role as the dominant technology in the transition toward fully electric transportation.

**Electric Range by Manufacturer**

In [ ]:
plt.figure(dpi=120)

ax = (
    battery.groupby('make')['electric_range']
    .mean()
    .sort_values(ascending=True)
    .tail(10)
    .plot(kind='barh', 
          figsize=(8,4), 
          color='#00d4aa',
          width=0.7)
)

ax.tick_params(axis='y', labelsize=7)
ax.tick_params(axis='x', labelsize=8)

plt.title('Most common Manufacturers by Average Electric Range', 
          fontsize=11,
          fontfamily='sans-serif')
plt.xlabel('Electric Range', 
           fontsize=9, 
           fontfamily='sans-serif')
plt.ylabel('Manufacturer',
           fontsize=9, 
           fontfamily='sans-serif')
plt.tight_layout()
plt.show()

* Jaguar shows the highest average electric range in the dataset, exceeding 175 miles per charge, followed by Wheego Electric Cars and Think, with average ranges of around 100 miles. In contrast, manufacturers such as Nissan and Tesla exhibit lower average ranges, approximately 60 miles and 52 miles, respectively. However, these results should be interpreted with caution, as they likely stem from the inclusion of older models and imbalances in the dataset — rather than the true performance of these manufacturers.

* It is also worth noting that recent models from Tesla, such as the Tesla Model 3 Long Range, can exceed 350 miles per charge. Therefore, the relatively lower average range observed for Tesla in this dataset is more indicative of data composition than current technological capability.

**Top Electric Vehicle Models by City (Registration Count)**

In [ ]:
plt.figure(dpi=120)

ax = (
    battery.groupby(['city','model'])
    .size()
    .sort_values(ascending=True)
    .tail(10)
    .plot(kind='barh',
          color= "#7DC6BE",
          figsize=(8,4),
          width=0.7
          )
    )

ax.tick_params(axis='y', labelsize=8)
ax.tick_params(axis='x', labelsize=8)

plt.title('Distribution of Leading EV Models Across Cities', 
          fontsize=11,
          fontfamily='sans-serif')
plt.xlabel('Count', 
           fontsize=9, 
           fontfamily='sans-serif')
plt.ylabel('City & Model',
           fontsize=9, 
           fontfamily='sans-serif')
plt.tight_layout()

* The results show that Tesla’s Model Y is the most frequently registered vehicle model across multiple cities, particularly in Seattle, and Bellevue. It consistently appears as the top model in several locations, indicating strong market dominance. Tesla Model 3 also ranks highly, especially in Seattle and Bellevue, while the Nissan Leaf remains one of the most common non-Tesla models. Overall, Tesla models, particularly Model Y, dominate EV registrations across major cities.

**County-Level Distribution of Leading EV Models**

In [ ]:
plt.figure(figsize=(8, 4), dpi=120)

ax = (
    battery.groupby(['county', 'model'])
    .size()
    .sort_values(ascending=True)
    .tail(10)
    .plot(kind='barh', width=0.7)
    )

ax.tick_params(axis='y', labelsize=8)
ax.tick_params(axis='x', labelsize=8)

plt.title('Distribution of Top EV Model in Counties', 
          fontsize=11, 
          fontfamily='sans-serif')
plt.ylabel('County & Model', 
           fontsize=9, 
           fontfamily='sans-serif')
plt.xlabel('Count', 
           fontsize=9, 
           fontfamily='sans-serif')

plt.tight_layout()
plt.show()

* Tesla’s Model Y dominates EV registrations across major counties, particularly in King County, with the Model 3 following closely behind.
* This pattern underscores Tesla’s strong regional market penetration, while the continued presence of models such as the Nissan Leaf and Chevrolet Bolt EV indicates that alternative, more affordable options still retain a meaningful share of the market.

**County-Level Distribution of BEVs and PHEVs**

In [ ]:
pd.DataFrame(battery.groupby(['county', 'electric_vehicle_type'])
             .size()
             .sort_values(ascending=False)
             .head(10))

* Battery Electric Vehicles (BEVs) dominate registrations across all major counties, with King County leading by a substantial margin. BEVs indicate a strong preference for fully electric vehicles over plug-in hybrids. While Plug-in Hybrid Electric Vehicles (PHEVs) are present, their counts are significantly lower. Overall, the data suggests that EV adoption across counties is driven primarily by BEVs, particularly in densely populated regions.

**Manufacturer-Level Distribution of BEVs and PHEVs**

In [ ]:
pd.DataFrame(
    battery.groupby(['make', 'electric_vehicle_type'])
             .size()
             .sort_values(ascending=False)
             .head(10)
             )

* Most leading manufacturers produce Battery Electric Vehicles (BEVs). Tesla overwhelmingly dominates  BEV registrations by a large margin. Other manufacturers such as Nissan, Chevrolet, Ford, and Hyundai also focus heavily on BEVs, but at much lower volumes compared to Tesla. In contrast, some manufacturers like Toyota and Jeep appear more prominently in the Plug-in Hybrid Electric Vehicle (PHEV) category. Overall, while BEVs dominate the market, manufacturers vary in their approach, with some focusing on fully electric vehicles and others maintaining a stronger presence in hybrid offerings.

**Trend of Electric Vehicle Types by Model Year (Registration Count)**

In [ ]:
pd.DataFrame(battery.groupby(['model_year', 'electric_vehicle_type'])
             .size()
             .sort_values(ascending=False)
             .head(30))

* The results show a clear upward trend in electric vehicle registrations over time, particularly for Battery Electric Vehicles (BEVs). BEVs dominate across recent model years, with a sharp increase from 2020 onwards and peaking in 2023 and 2024. This reflects rapid growth in fully electric vehicle adoption and improvements in EV technology. While Plug-in Hybrid Electric Vehicles (PHEVs) are also present, their growth is more moderate and consistently lower than BEVs across all years. Overall, the data indicates a strong shift toward fully electric vehicles in recent years.

## 7. Feature Relationships


In [ ]:
battery[['model_year','electric_range']].corr()

* The correlation matrix shows a moderate negative relationship between model_year and electric_range (correlation ≈ -0.54). This indicates that, within this dataset, higher model years are generally associated with lower electric range. In other words, as model_year increases, electric_range tends to decrease on average. However, this is a statistical trend rather than a deterministic rule. This finding is somewhat counterintuitive, as technological advancements would typically be expected to increase electric range over time.

In [ ]:
 plt.figure(figsize=(6, 4), dpi=120)

ax = sns.scatterplot(x=battery.model_year, 
                     y=battery.electric_range, 
                     color='#00d4aa', 
                     edgecolor='black',
                     linewidth=0.2
                     )

ax.tick_params(axis='y', labelsize=8)
ax.tick_params(axis='x', labelsize=8)

plt.title('Relationship between Model Year and Electric Range',
          fontsize=11,
          fontfamily='sans-serif')

plt.ylabel('Electric Range',
           fontsize=9,
           fontfamily='sans-serif')
plt.xlabel('Model Year',
           fontsize=9,
           fontfamily='sans-serif')
plt.tight_layout()
plt.show()

* To further investigate this relationship, a scatter plot of model_year versus electric_range was generated. The plot reveals that the relationship is not strictly linear. While the correlation suggests that newer model years are associated with lower electric range, the plot shows a more complex pattern: recent years (around 2018–2025) contain a wide spread of ranges, from very low (near 0–50) to very high (200–330+).

* This indicates that the negative correlation is likely driven by data clustering and imbalance rather than a true downward trend. In earlier years, there are fewer data points but some relatively high-range vehicles, while in newer years there are many more low-range vehicles (possibly hybrids, compliance EVs, or shorter-range models), which pulls the overall correlation downward. At the same time, high-range vehicles still exist in recent years, showing that technology has improved. Moreover, another reason is that the dataset includes a broader mix of vehicle types.

* Overall, the plot suggests that model year alone is not a strong predictor of electric range.

## Key Insights

* BEVs dominate the EV market
* Tesla is the leading manufacturer
* EV adoption has accelerated after 2020
* Registrations are concentrated in a few counties

## 8. Conclusion

The exploratory analysis highlights several key patterns in the electric vehicle market. Battery Electric Vehicles (BEVs) dominate registrations, significantly outpacing Plug-in Hybrid Electric Vehicles (PHEVs), indicating a strong shift toward fully electric mobility. Tesla emerges as the clear market leader, with models such as the Model Y and Model 3 accounting for a substantial share of total registrations.

EV adoption has accelerated in recent model years, reflecting advancements in technology and increasing consumer demand. Additionally, registrations are highly concentrated in a few counties, suggesting that regional factors—such as infrastructure availability and policy support—play a critical role in driving adoption.

However, the dataset presents limitations, particularly the large “eligibility unknown” category for CAFV, which restricts deeper interpretation. Despite this, the analysis provides a clear overview of current EV trends and underscores the key factors influencing the transition toward electric mobility.

## 9. References

Data Government. 2026. *Electric Vehicle Population Data*. Retrived from https://catalog.data.gov/dataset/electric-vehicle-population-data